# Fase 7: Análisis literario y validación emocional mediante IA generativa
---
Este cuaderno introduce una capa cualitativa al sistema utilizando inteligencia artificial generativa.

Los objetivos son:
1. **Generación de resúmenes:** Actuar como un crítico literario para redactar un párrafo explicativo sobre el significado de cada canción.
2. **Validación/Auditoría:** Recibir la emoción matemática detectada previamente por `samlowe/roberta-base-go_emotions` y ejercer de "auditor", razonando si esa emoción es correcta dada la letra de la canción.

In [ ]:
import pandas as pd
import time
import os

from tqdm.notebook import tqdm
from dotenv import load_dotenv
from groq import Groq

import warnings
warnings.filterwarnings('ignore')

In [ ]:
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
for model in client.models.list().data:
    print(f"NOMBRE (ID): {model.id}\nCREADOR: {model.owned_by}\n" + "-"*50)

NOMBRE (ID): openai/gpt-oss-120b
CREADOR: OpenAI
--------------------------------------------------
NOMBRE (ID): whisper-large-v3
CREADOR: OpenAI
--------------------------------------------------
NOMBRE (ID): llama-3.1-8b-instant
CREADOR: Meta
--------------------------------------------------
NOMBRE (ID): meta-llama/llama-prompt-guard-2-22m
CREADOR: Meta
--------------------------------------------------
NOMBRE (ID): allam-2-7b
CREADOR: SDAIA
--------------------------------------------------
NOMBRE (ID): openai/gpt-oss-20b
CREADOR: OpenAI
--------------------------------------------------
NOMBRE (ID): groq/compound-mini
CREADOR: Groq
--------------------------------------------------
NOMBRE (ID): openai/gpt-oss-safeguard-20b
CREADOR: OpenAI
--------------------------------------------------
NOMBRE (ID): meta-llama/llama-4-scout-17b-16e-instruct
CREADOR: Meta
--------------------------------------------------
NOMBRE (ID): whisper-large-v3-turbo
CREADOR: OpenAI
-----------------------

In [5]:
df = pd.read_csv('../data/processed/taylor_swift_metrics.csv')

### 7.1 Modelo `llama-3.1-8b-instant`

In [6]:
meanings = []
validations_ai = []

def retry_meaning(prompt, model='llama-3.1-8b-instant', max_retries=4):
    wait = 15
    for i in range(max_retries):
        try:
            answer = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are an expert music analyst and literary critic."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=200
            )
            return answer.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                print(f"Límite rozado, pausando {wait}s...")
                time.sleep(wait)
                wait *= 2
            else:
                return f"Error, {str(e)}"
    return "Error por cuota tras múltiples reintentos."

def retry_validation(prompt, model='llama-3.1-8b-instant', max_retries=4):
    wait = 15
    for i in range(max_retries):
        try:
            answer = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are an expert AI auditor and music emotion analyst."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=200
            )
            return answer.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                print(f"Límite rozado, pausando {wait}s...")
                time.sleep(wait)
                wait *= 2
            else:
                return f"Error, {str(e)}"
    return "Error por cuota tras múltiples reintentos."

for idx, row in tqdm(df.iterrows(), total=len(df)):
    artist = "Taylor Swift"
    title = row['title']
    album = row['album']
    lyrics = str(row['lyrics_full'])[:1500]
    theme = row['theme_zeroshot']
    emotion = row['emotion_goemotions']

    prompt_meaning = f"""
    Analyze the following lyrics from the song '{title}' from the album '{album}' by {artist}:
    "{lyrics}"

    Write a single, brief paragraph (maximum 5 lines) IN SPANISH explaining the story behind this song and its main message.

    Respond strictly IN SPANISH.
    """

    prompt_validation = f"""
    Evaluate the accuracy of a Machine Learning classification for the song '{title}' by {artist}.

    - Lyrics: "{lyrics}"
    - Predicted emotion by the ML model: {emotion}
    - Predited theme by the ML model: {theme}

    Write a single, brief paragraph (maximum 5 lines) IN SPANISH explaining if this emotion is accurate for the song's narrative.
    If it is not accurate, suggest a better emotion and briefly explain why, basing your explanation on the lyrics of the song.

    Respond strictly IN SPANISH.
    """

    answer_meaning = retry_meaning(prompt_meaning)
    meanings.append(answer_meaning)
    time.sleep(3)

    answer_validation = retry_validation(prompt_validation)
    validations_ai.append(answer_validation)
    time.sleep(3)

  0%|          | 0/242 [00:00<?, ?it/s]

In [7]:
df['llm_meaning'] = meanings
df['llm_validation'] = validations_ai

In [8]:
print("\n" + "="*80)
print(f"CANCIÓN: {df.iloc[188]['title']}")
print(f"SIGNIFICADO: \n{df.iloc[188]['llm_meaning']}\n")
print(f"VALIDACIÓN IA: \n{df.iloc[188]['llm_validation']}")
print("="*80)

print("\n" + "="*80)
print(f"CANCIÓN: {df.iloc[111]['title']}")
print(f"SIGNIFICADO: \n{df.iloc[111]['llm_meaning']}\n")
print(f"VALIDACIÓN IA: \n{df.iloc[111]['llm_validation']}")
print("="*80)

print("\n" + "="*80)
print(f"CANCIÓN: {df.iloc[23]['title']}")
print(f"SIGNIFICADO: \n{df.iloc[23]['llm_meaning']}\n")
print(f"VALIDACIÓN IA: \n{df.iloc[23]['llm_validation']}")
print("="*80)


CANCIÓN: Haunted (Taylor’s Version)
SIGNIFICADO: 
La canción "Haunted" de Taylor Swift narra una historia de amor que se desmorona, donde la protagonista se siente abandonada y confundida después de que su pareja se alejó. La letra destaca la sensación de pérdida y desesperanza que se siente al perder a alguien a quien se quería con todo el corazón. El mensaje principal de la canción es que el amor puede ser una experiencia dolorosa, pero es imposible olvidarlo. La cantante refleja su lucha interna para aceptar la realidad del fin del amor y la imposibilidad de volver atrás. En última instancia, la canción es un canto a la memoria del amor perdido.

VALIDACIÓN IA: 
La predicción del modelo de ML como "desconocido" no se ajusta a la narrativa de la canción "Haunted (Taylor's Versión)" de Taylor Swift. La canción describe una ruptura emocional y un sentimiento de pérdida, por lo que un emoción más adecuada sería la tristeza o la melancolía. Esto se refleja en las letras, donde Taylor Sw

### 7.2 Modelo `gemma2-9b-it`

In [10]:
meanings = []
validations_ai = []

def retry_meaning(prompt, model='gemma2-9b-it', max_retries=4):
    wait = 15
    for i in range(max_retries):
        try:
            answer = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are an expert music analyst and literary critic."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=200
            )
            return answer.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                print(f"Límite rozado, pausando {wait}s...")
                time.sleep(wait)
                wait *= 2
            else:
                return f"Error, {str(e)}"
    return "Error por cuota tras múltiples reintentos."

def retry_validation(prompt, model='gemma2-9b-it', max_retries=4):
    wait = 15
    for i in range(max_retries):
        try:
            answer = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are an expert AI auditor and music emotion analyst."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=200
            )
            return answer.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                print(f"Límite rozado, pausando {wait}s...")
                time.sleep(wait)
                wait *= 2
            else:
                return f"Error, {str(e)}"
    return "Error por cuota tras múltiples reintentos."

for idx, row in tqdm(df.iterrows(), total=len(df)):
    artist = "Taylor Swift"
    title = row['title']
    album = row['album']
    lyrics = str(row['lyrics_full'])[:1500]
    theme = row['theme_zeroshot']
    emotion = row['emotion_goemotions']

    prompt_meaning = f"""
    Analyze the following lyrics from the song '{title}' from the album '{album}' by {artist}:
    "{lyrics}"

    Write a single, brief paragraph (maximum 5 lines) IN SPANISH explaining the story behind this song and its main message.

    Respond strictly IN SPANISH.
    """

    prompt_validation = f"""
    Evaluate the accuracy of a Machine Learning classification for the song '{title}' by {artist}.

    - Lyrics: "{lyrics}"
    - Predicted emotion by the ML model: {emotion}
    - Predited theme by the ML model: {theme}

    Write a single, brief paragraph (maximum 5 lines) IN SPANISH explaining if this emotion is accurate for the song's narrative.
    If it is not accurate, suggest a better emotion and briefly explain why, basing your explanation on the lyrics of the song.

    Respond strictly IN SPANISH.
    """

    answer_meaning = retry_meaning(prompt_meaning)
    meanings.append(answer_meaning)
    time.sleep(3)

    answer_validation = retry_validation(prompt_validation)
    validations_ai.append(answer_validation)
    time.sleep(3)

  0%|          | 0/242 [00:00<?, ?it/s]

In [11]:
df['llm_meaning_2'] = meanings
df['llm_validation_2'] = validations_ai

In [12]:
print("\n" + "="*80)
print(f"CANCIÓN: {df.iloc[188]['title']}")
print(f"SIGNIFICADO: \n{df.iloc[188]['llm_meaning_2']}\n")
print(f"VALIDACIÓN IA: \n{df.iloc[188]['llm_validation_2']}")
print("="*80)

print("\n" + "="*80)
print(f"CANCIÓN: {df.iloc[111]['title']}")
print(f"SIGNIFICADO: \n{df.iloc[111]['llm_meaning_2']}\n")
print(f"VALIDACIÓN IA: \n{df.iloc[111]['llm_validation_2']}")
print("="*80)

print("\n" + "="*80)
print(f"CANCIÓN: {df.iloc[23]['title']}")
print(f"SIGNIFICADO: \n{df.iloc[23]['llm_meaning_2']}\n")
print(f"VALIDACIÓN IA: \n{df.iloc[23]['llm_validation_2']}")
print("="*80)


CANCIÓN: Haunted (Taylor’s Version)
SIGNIFICADO: 
Error, Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

VALIDACIÓN IA: 
Error, Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

CANCIÓN: I Almost Do (Taylor’s Version)
SIGNIFICADO: 
Error, Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error'

### 7.3 Modelo `moonshotai/kimi-k2-instruct`

In [13]:
meanings = []
validations_ai = []

def retry_meaning(prompt, model='moonshotai/kimi-k2-instruct', max_retries=4):
    wait = 15
    for i in range(max_retries):
        try:
            answer = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are an expert music analyst and literary critic."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=200
            )
            return answer.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                print(f"Límite rozado, pausando {wait}s...")
                time.sleep(wait)
                wait *= 2
            else:
                return f"Error, {str(e)}"
    return "Error por cuota tras múltiples reintentos."

def retry_validation(prompt, model='moonshotai/kimi-k2-instruct', max_retries=4):
    wait = 15
    for i in range(max_retries):
        try:
            answer = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are an expert AI auditor and music emotion analyst."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=200
            )
            return answer.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                print(f"Límite rozado, pausando {wait}s...")
                time.sleep(wait)
                wait *= 2
            else:
                return f"Error, {str(e)}"
    return "Error por cuota tras múltiples reintentos."

for idx, row in tqdm(df.iterrows(), total=len(df)):
    artist = "Taylor Swift"
    title = row['title']
    album = row['album']
    lyrics = str(row['lyrics_full'])[:1500]
    theme = row['theme_zeroshot']
    emotion = row['emotion_goemotions']

    prompt_meaning = f"""
    Analyze the following lyrics from the song '{title}' from the album '{album}' by {artist}:
    "{lyrics}"

    Write a single, brief paragraph (maximum 5 lines) IN SPANISH explaining the story behind this song and its main message.

    Respond strictly IN SPANISH.
    """

    prompt_validation = f"""
    Evaluate the accuracy of a Machine Learning classification for the song '{title}' by {artist}.

    - Lyrics: "{lyrics}"
    - Predicted emotion by the ML model: {emotion}
    - Predited theme by the ML model: {theme}

    Write a single, brief paragraph (maximum 5 lines) IN SPANISH explaining if this emotion is accurate for the song's narrative.
    If it is not accurate, suggest a better emotion and briefly explain why, basing your explanation on the lyrics of the song.

    Respond strictly IN SPANISH.
    """

    answer_meaning = retry_meaning(prompt_meaning)
    meanings.append(answer_meaning)
    time.sleep(3)

    answer_validation = retry_validation(prompt_validation)
    validations_ai.append(answer_validation)
    time.sleep(3)

  0%|          | 0/242 [00:00<?, ?it/s]

In [14]:
df['llm_meaning_3'] = meanings
df['llm_validation_3'] = validations_ai

In [15]:
print("\n" + "="*80)
print(f"CANCIÓN: {df.iloc[188]['title']}")
print(f"SIGNIFICADO: \n{df.iloc[188]['llm_meaning_3']}\n")
print(f"VALIDACIÓN IA: \n{df.iloc[188]['llm_validation_3']}")
print("="*80)

print("\n" + "="*80)
print(f"CANCIÓN: {df.iloc[111]['title']}")
print(f"SIGNIFICADO: \n{df.iloc[111]['llm_meaning_3']}\n")
print(f"VALIDACIÓN IA: \n{df.iloc[111]['llm_validation_3']}")
print("="*80)

print("\n" + "="*80)
print(f"CANCIÓN: {df.iloc[23]['title']}")
print(f"SIGNIFICADO: \n{df.iloc[23]['llm_meaning_3']}\n")
print(f"VALIDACIÓN IA: \n{df.iloc[23]['llm_validation_3']}")
print("="*80)


CANCIÓN: Haunted (Taylor’s Version)
SIGNIFICADO: 
La canción narra la caída repentina de una relación que parecía segura: la protagonista se queda paralizada ante la marcha imprevista de su pareja y se niega a aceptar el final, aferrándose a los recuerdos y a cada palabra pronunciada. Aunque intenta sustituirlo con alguien más, solo desea que sea él, y el vacío que deja lo convierte en un fantasma que la persigue, repitiendo “no puedo volver atrás, estoy embrujada” como certeza de que el daño ya es irreversible.

VALIDACIÓN IA: 
La emoción «unknown» no refleja la canción: el texto late de angustia, desamparo y obsesión por una relación rota (“can’t breathe whenever you’re gone / I’m haunted”).  
Una etiqueta más precisa es angustia o desesperación, porque la narradora se siente literalmente perseguida por la ausencia y no logra desprenderse del recuerdo.

CANCIÓN: I Almost Do (Taylor’s Version)
SIGNIFICADO: 
Error, Error code: 503 - {'error': {'message': 'moonshotai/kimi-k2-instruct i

### 7.4 Modelo `meta-llama/llama-4-scout-17b-16e-instruct`

In [16]:
meanings = []
validations_ai = []

def retry_meaning(prompt, model='meta-llama/llama-4-scout-17b-16e-instruct', max_retries=4):
    wait = 15
    for i in range(max_retries):
        try:
            answer = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are an expert music analyst and literary critic."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=200
            )
            return answer.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                print(f"Límite rozado, pausando {wait}s...")
                time.sleep(wait)
                wait *= 2
            else:
                return f"Error, {str(e)}"
    return "Error por cuota tras múltiples reintentos."

def retry_validation(prompt, model='meta-llama/llama-4-scout-17b-16e-instruct', max_retries=4):
    wait = 15
    for i in range(max_retries):
        try:
            answer = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are an expert AI auditor and music emotion analyst."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=200
            )
            return answer.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                print(f"Límite rozado, pausando {wait}s...")
                time.sleep(wait)
                wait *= 2
            else:
                return f"Error, {str(e)}"
    return "Error por cuota tras múltiples reintentos."

for idx, row in tqdm(df.iterrows(), total=len(df)):
    artist = "Taylor Swift"
    title = row['title']
    album = row['album']
    lyrics = str(row['lyrics_full'])[:1500]
    theme = row['theme_zeroshot']
    emotion = row['emotion_goemotions']

    prompt_meaning = f"""
    Analyze the following lyrics from the song '{title}' from the album '{album}' by {artist}:
    "{lyrics}"

    Write a single, brief paragraph (maximum 5 lines) IN SPANISH explaining the story behind this song and its main message.

    Respond strictly IN SPANISH.
    """

    prompt_validation = f"""
    Evaluate the accuracy of a Machine Learning classification for the song '{title}' by {artist}.

    - Lyrics: "{lyrics}"
    - Predicted emotion by the ML model: {emotion}
    - Predited theme by the ML model: {theme}

    Write a single, brief paragraph (maximum 5 lines) IN SPANISH explaining if this emotion is accurate for the song's narrative.
    If it is not accurate, suggest a better emotion and briefly explain why, basing your explanation on the lyrics of the song.

    Respond strictly IN SPANISH.
    """

    answer_meaning = retry_meaning(prompt_meaning)
    meanings.append(answer_meaning)
    time.sleep(3)

    answer_validation = retry_validation(prompt_validation)
    validations_ai.append(answer_validation)
    time.sleep(3)

  0%|          | 0/242 [00:00<?, ?it/s]

In [17]:
df['llm_meaning_4'] = meanings
df['llm_validation_4'] = validations_ai

In [18]:
print("\n" + "="*80)
print(f"CANCIÓN: {df.iloc[188]['title']}")
print(f"SIGNIFICADO: \n{df.iloc[188]['llm_meaning_4']}\n")
print(f"VALIDACIÓN IA: \n{df.iloc[188]['llm_validation_4']}")
print("="*80)

print("\n" + "="*80)
print(f"CANCIÓN: {df.iloc[111]['title']}")
print(f"SIGNIFICADO: \n{df.iloc[111]['llm_meaning_4']}\n")
print(f"VALIDACIÓN IA: \n{df.iloc[111]['llm_validation_4']}")
print("="*80)

print("\n" + "="*80)
print(f"CANCIÓN: {df.iloc[23]['title']}")
print(f"SIGNIFICADO: \n{df.iloc[23]['llm_meaning_4']}\n")
print(f"VALIDACIÓN IA: \n{df.iloc[23]['llm_validation_4']}")
print("="*80)


CANCIÓN: Haunted (Taylor’s Version)
SIGNIFICADO: 
La canción "Haunted (Taylor's Version)" describe una relación rota y la desesperación de la narradora por aferrarse a lo que queda. La letra revela una sensación de pérdida y confusión tras la partida de la otra persona. La narradora lucha por aceptar la realidad y se siente "embrujada" por los recuerdos de la relación. El mensaje principal es la intensidad del dolor y la nostalgia que persisten incluso después del final. La canción es un lamento por lo perdido.

VALIDACIÓN IA: 
La emoción predicha por el modelo de Machine Learning no se proporciona, pero basándome en las letras de la canción "Haunted (Taylor's Version)" de Taylor Swift, puedo inferir que la emoción principal expresada es la tristeza y la desesperanza tras una ruptura. El tono de la canción refleja una sensación de pérdida y añoranza, con frases como "Can't breathe whenever you're gone" y "I'm haunted". La emoción de tristeza y desolación parece ser la más precisa para

In [ ]:
df.to_csv('../data/processed/taylor_swift_definitivo2.csv', index=False)

### Conclusión del benchmark
Tras someter los 4 LLM al mismo *prompt* y evaluar sus resultados, se han extraído las siguientes decisiones de diseño:
* Aunque modelos pesados como `meta-llama/llama-4-scout-17b-16e-instruct` ofrecen gran riqueza léxica, sus tiempos de respuesta en bucles de más de 250 iteraciones no son viables para la fase de procesamiento masivo.
* Se descartó el modelo `gemma2-9b-it` porque ha sido discontinuado por Groq, y también el modelo `moonshotai/kimi-k2-instruct` por ser inestable en este caso de uso, lanzando errores de saturación del servidor al intentar procesar las peticiones en lote.
* El modelo ganador fue `llama-3.1-8b-instant` porque ofrece un equilibrio entre velocidad, capacidad de seguir instrucciones de formato y calidad literaria.
